# Analises Gold (Curated Views)

## 1. Carregamento das tabelas Gold
Carrega a fato e as dimensões da camada Gold e realiza a conferência inicial.

In [0]:
from pyspark.sql import functions as F

catalog = "vcredit"
gold = f"{catalog}.gold"

df_fact = spark.table(f"{gold}.fact_chamados")
df_motivo = spark.table(f"{gold}.dim_motivo")
df_canal = spark.table(f"{gold}.dim_canal")
df_cliente = spark.table(f"{gold}.dim_cliente")
df_atendente = spark.table(f"{gold}.dim_atendente")
df_data = spark.table(f"{gold}.dim_data")

print("Tabelas carregadas com sucesso.")

## 2. Base detalhada com as dimensões
Centraliza as regras acordadas:
- CSAT: Notas de 1 a 5, considerando 4 e 5 como satisfeitos.
- Custos: Mantém valores nulos. Cria flag de cobertura.
- Experiência: Tempo de empresa exato na data da ocorrência do chamado.

In [0]:
df_detalhado = (
    df_fact.alias("f")
    .join(df_motivo.alias("m"), "id_motivo", "left")
    .join(df_canal.alias("c"), "id_canal", "left")
    .join(df_cliente.alias("cli"), "id_cliente", "left")
    .join(df_atendente.alias("a"), "id_atendente", "left")
    .join(df_data.alias("d"), F.col("f.id_data") == F.col("d.id_tempo"), "left")
    .select(
        F.col("f.id_chamado"),
        F.col("d.data"), F.col("d.mes"), F.col("d.ano"), F.col("d.dia_semana"),
        F.col("m.nome_motivo"), F.col("m.categoria"), F.col("m.criticidade"),
        F.col("c.nome_canal"), F.col("c.tipo_canal"),
        F.col("cli.faixa_etaria"), F.col("cli.nome_cliente"),
        F.col("a.nome_atendente"), F.col("a.nivel_atendimento"),
        F.datediff(F.col("d.data"), F.col("a.data_admissao")).alias("experiencia_dias_no_atendimento"),
        F.col("f.resolvido"),
        F.col("f.custo"),
        F.when(F.col("f.custo").isNotNull(), 1).otherwise(0).alias("flag_custo_valido"),
        F.col("f.nota_atendimento"),
        F.when(F.col("f.nota_atendimento").between(1, 5), 1).otherwise(0).alias("flag_nota_valida"),
        F.when(F.col("f.nota_atendimento").isin(4, 5), 1).otherwise(0).alias("flag_satisfeito"),
        F.col("f.tempo_espera_seg"),
        F.col("f.tempo_atendimento_seg")
    )
)

df_detalhado.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_chamados_detalhado")

## 3. Indicadores mensais e financeiros
Visão macro: Onde e por que o custo cresce e como evolui a satisfação global ao longo dos meses.

In [0]:
df_kpis_mensal = (
    df_detalhado
    .groupBy("ano", "mes")
    .agg(
        F.count("id_chamado").alias("volume_total"),
        F.sum("custo").alias("custo_total_registrado"),
        F.round(F.avg("custo"), 2).alias("custo_medio_observado"),
        F.round(F.sum("flag_custo_valido") * 100 / F.count("id_chamado"), 2).alias("cobertura_custos_pct"),
        F.round(F.sum(F.when(F.col("resolvido") == "S", 1).otherwise(0)) * 100 / F.count("id_chamado"), 2).alias("taxa_resolucao_pct"),
        F.round(F.sum("flag_satisfeito") * 100 / F.sum("flag_nota_valida"), 2).alias("csat_pct"),
        F.round(F.avg("tempo_espera_seg"), 2).alias("espera_media_seg")
    )
)
df_kpis_mensal.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_kpis_mensal")

## 4. Desempenho por motivo e criticidade

In [0]:
df_desempenho_motivo = (
    df_detalhado
    .groupBy("categoria", "nome_motivo", "criticidade")
    .agg(
        F.count("id_chamado").alias("volume"),
        F.sum("custo").alias("custo_total"),
        F.round(F.avg("custo"), 2).alias("custo_medio"),
        F.round(F.sum("flag_satisfeito") * 100 / F.sum("flag_nota_valida"), 2).alias("csat_pct"),
        F.round(F.sum(F.when(F.col("resolvido") == "S", 1).otherwise(0)) * 100 / F.count("id_chamado"), 2).alias("resolucao_pct")
    )
)
df_desempenho_motivo.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_desempenho_motivo")

## 5. Desempenho por canal
Eficiência do atendimento e perdas de funil em cada meio.

In [0]:
df_desempenho_canal = (
    df_detalhado
    .groupBy("tipo_canal", "nome_canal")
    .agg(
        F.count("id_chamado").alias("volume"),
        F.sum(F.when(F.col("resolvido") == "NAO_ATENDIDO", 1).otherwise(0)).alias("desistencia_ou_falha"),
        F.round(F.sum("flag_satisfeito") * 100 / F.sum("flag_nota_valida"), 2).alias("csat_pct"),
        F.round(F.avg("tempo_espera_seg"), 2).alias("espera_media_seg")
    )
)
df_desempenho_canal.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_desempenho_canal")

## 6. Experiência e desempenho dos atendentes
Relação entre a maturidade na função e as métricas de tempo/resolução.

In [0]:
df_desempenho_atendente = (
    df_detalhado
    .filter(F.col("nome_atendente").isNotNull())
    .withColumn("faixa_experiencia", 
        F.when(F.col("experiencia_dias_no_atendimento") < 90, "Até 3 meses")
         .when(F.col("experiencia_dias_no_atendimento") <= 365, "3 a 12 meses")
         .otherwise("Mais de 1 ano")
    )
    .groupBy("nivel_atendimento", "faixa_experiencia")
    .agg(
        F.count("id_chamado").alias("volume"),
        F.round(F.avg("tempo_atendimento_seg"), 2).alias("tma_medio_seg"),
        F.round(F.sum(F.when(F.col("resolvido") == "S", 1).otherwise(0)) * 100 / F.count("id_chamado"), 2).alias("resolucao_pct")
    )
)
df_desempenho_atendente.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_desempenho_atendente")

## 7. Demanda por dia da semana

In [0]:
df_demanda_dia = (
    df_detalhado
    .groupBy("dia_semana")
    .agg(
        F.count("id_chamado").alias("volume"),
        F.round(F.avg("tempo_espera_seg"), 2).alias("espera_media_seg"),
        F.sum(F.when(F.col("resolvido") == "NAO_ATENDIDO", 1).otherwise(0)).alias("volume_abandonado")
    )
)
df_demanda_dia.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_demanda_dia_semana")

## 8. Autosserviço por faixa etária

In [0]:
df_autosservico = (
    df_detalhado
    .filter(F.col("tipo_canal").isin("Autosserviço", "Digital", "URA", "Chatbot", "App", "Site"))
    .groupBy("nome_canal", "faixa_etaria")
    .agg(
        F.count("id_chamado").alias("volume"),
        F.round(F.sum(F.when(F.col("resolvido") == "S", 1).otherwise(0)) * 100 / F.count("id_chamado"), 2).alias("resolucao_pct")
    )
)
df_autosservico.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_autosservico_faixa_etaria")

## 9. Qualidade e integridade dos dados

In [0]:
df_qualidade = (
    df_detalhado
    .select(
        F.sum(F.when(F.col("custo").isNull(), 1).otherwise(0)).alias("custos_nulos"),
        F.sum(F.when(F.col("flag_nota_valida") == 0, 1).otherwise(0)).alias("notas_invalidas"),
        F.sum(F.when(F.col("nome_cliente").isNull() | (F.col("nome_cliente") == "Desconhecido"), 1).otherwise(0)).alias("clientes_orfaos")
    )
)
df_qualidade.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_qualidade_dados")

## 10. Análises cruzadas e priorização de motivos
Cria matrizes para investigar ofensores combinados de canal x motivo.

In [0]:
df_priorizacao = (
    spark.table(f"{gold}.vw_desempenho_motivo")
    .orderBy(F.desc("custo_total"), F.asc("csat_pct"))
)
df_priorizacao.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_priorizacao_motivos")

# Bloco 12: Motivo x Canal
df_motivo_canal = (
    df_detalhado
    .groupBy("nome_motivo", "nome_canal")
    .agg(
        F.count("id_chamado").alias("volume"),
        F.round(F.avg("custo"), 2).alias("custo_medio"),
        F.round(F.sum("flag_satisfeito") * 100 / F.sum("flag_nota_valida"), 2).alias("csat_pct")
    )
)
df_motivo_canal.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_motivo_canal")

## 11. Indicadores financeiros para comparação
Permite analisar economia esperada vs observada, cobertura financeira e custo por resolução.

In [0]:
df_financeiro = (
    df_detalhado
    .filter(F.col("flag_custo_valido") == 1)
    .groupBy("ano", "mes")
    .agg(
        F.sum("custo").alias("custo_total"),
        F.round(F.sum("custo") / F.sum(F.when(F.col("resolvido") == "S", 1).otherwise(0)), 2).alias("custo_por_resolucao"),
        F.count("id_chamado").alias("vol_chamados_precificados")
    )
)
df_financeiro.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.vw_indicadores_financeiros_mensais")